###Bibliotheken einfügen


In [ ]:
# import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import requests

###Goldpreis von Alpha Vantage ziehen


In [ ]:
# Dein API-Schlüssel von Alpha Vantage
api_key = 'JX03FK2W87N81TEN'

# API-Endpunkt für den täglichen Goldpreis (XAU/USD)
url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=XAUUSD&apikey={api_key}&outputsize=full'

 # API-Anfrage senden
response = requests.get(url)
data = response.json()

# Die historischen Daten aus der Antwort extrahieren
if "Time Series (Daily)" in data:   
    time_series = data["Time Series (Daily)"]

    # Die Daten in ein pandas DataFrame umwandeln
    df = pd.DataFrame.from_dict(time_series, orient='index')
    df = df.astype(float)  # Umwandeln der Daten in numerische Werte
    df.index = pd.to_datetime(df.index)  # Index als Datum umwandeln

    # Speichern der Daten als CSV-Datei
    df.to_csv('gold_price.csv')
    print('Daten wurden erfolgreich als CSV gespeichert.')
    
# Ausgabe der ersten Zeilen des DataFrames
    print(df.head())
    print(len(df))
    print(df.index.min())  # Ältestes Datum
    print(df.index.max())  # Neuestes Datum

else:
    print("Fehlerhafte API-Antwort:")
    print(data)
    


##Vorverarbeitung

In [ ]:
# Vorverarbeitung und filtern
df_goldpreis=pd.read_csv('gold_prices.csv') # einlesen
df_goldpreis = df_goldpreis.rename(columns={'4. close': 'Goldpreis_am_Tag'})
df_goldpreis = df_goldpreis['Goldpreis_am_Tag'] #Spalten löschen
df_goldpreis.index = pd.to_datetime(df.index) #Datum umwandeln
df_goldpreis = df_goldpreis[df_goldpreis.index > pd.Timestamp('2021-01-01')] # nach 2021 filtern

# Da Datenpunkte an Wochenenden nicht plausibel sind, wurden sie gelöscht
# print(df_goldpreis.index.dayofweek.value_counts())
df_goldpreis = df_goldpreis[df_goldpreis.index.dayofweek < 5]
# print(len(df_goldpreis))


##Vollständigkeitsprüfung. Die Daten liegen erst ab 2021 vollständig vor.

In [ ]:
# Die Daten werden auf Vollständigkeit abgeglichen mit allen möglichen Werktagen
# Dazu werden alle Werktage seit 2021 in eine Liste gespeichert
Werktage_seit_2021 = pd.date_range(start='1/1/2021', end='12/05/2025', freq='D')
Werktage_seit_2021 = Werktage_seit_2021[Werktage_seit_2021.dayofweek < 5]
Werktage_seit_2021 = Werktage_seit_2021.normalize()
print(len(Werktage_seit_2021))

# Überprüfung, für welche Werktage keine Daten vorliegen
counter=0
Fehlwerte=[]
for i in Werktage_seit_2021:
    if i not in df_goldpreis.index.normalize():
        counter += 1
        Fehlwerte.append(i)
        # if i < pd.Timestamp('2021-01-01'):
            # print(i) 
            ## --> 185 Fehlwerte in 2020
            # counter_2020 += 1

print(counter)

# Abgleich mit den Werktagen, an denen die Börse geschlossen war
import pandas_market_calendars as mcal

# NYSE-Kalender laden
nyse = mcal.get_calendar('NYSE')

# Handelszeitplan abrufen (enthält nur Handelstage)
schedule = nyse.schedule(start_date='2021-01-01', end_date='2024-12-31')

richtige_Fehlwerte=[]
falsche_Fehlwerte=[]
for i in Fehlwerte:
    if i not in schedule: 
        richtige_Fehlwerte.append(i)
    elif i in schedule:
        falsche_Fehlwerte.append(i)

print('Fehlwerte an Schließtagen: ' + str(len(richtige_Fehlwerte))) 
print('Wirkliche Fehlwerte: ' + str (len(falsche_Fehlwerte)))

# Fazit: Es liegen für alle Werktage, an denen die Börse geöffnet war, Daten vor.

###Maximalwertuntersuchung und Outlier Detection 

In [ ]:
# Boxplot für Überblick
plt.boxplot(df_goldpreis)
plt.title('Boxplot Goldpreis')
plt.show()
plt.clf()

#Ausreißer finden
def find_outliers_IQR(df):
   q1=df.quantile(0.25)
   q3=df.quantile(0.75)
   IQR=q3-q1
   outliers = df[((df<(q1-1.5*IQR)) | (df>(q3+1.5*IQR)))]
   return outliers

print('Anzahl an Outliern:' + str(len(find_outliers_IQR(df_goldpreis))))
print(find_outliers_IQR(df_goldpreis))
# --> Online Abgleich: nur plausible Werte

plt.title('Verteilung des Goldpreises seit 2021')
plt.hist(df_goldpreis, bins=50)
plt.show()
plt.clf()

##Goldpreis im Zeitverlauf

In [ ]:
# Goldpreisentwicklung
ax = plt.subplot(1, 1, 1)
plt.title('Goldpreisentwicklung seit 2021')
plt.plot(df_goldpreis)
plt.xticks(rotation=45)
plt.show()
plt.clf()